In [1]:
# ============ CELLA 1 — Config ============
import os, re, asyncio, logging
import pandas as pd, tiktoken
from openai import AsyncOpenAI

# La chiave NON è nel codice. Impostala in una cella PRIMA di lanciare:
#   os.environ["AZURE_OPENAI_API_KEY"] = "la-tua-chiave"
with open("Other/key.txt") as f:
    os.environ["AZURE_OPENAI_API_KEY"] = f.read().strip()
    
ENDPOINT     = "https://daily-digest-mini-resource.services.ai.azure.com/openai/v1"
MODEL        = "gpt-5-mini"
CSV_IN       = "Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv"
CAP          = None          # stesso budget dei fine-tuned; None = articolo intero
OUT          = f"preds_gpt5mini_cap{CAP}.csv"
LOG          = f"run_gpt5mini_cap{CAP}.log"
CONCURRENCY  = 12           # abbassa a 6-8 se nel log vedi molti 429
MAX_ATTEMPTS = 5            # tentativi per singolo articolo
REQ_TIMEOUT  = 120          # secondi max per chiamata
SAVE_EVERY   = 200          # scrive il checkpoint ogni N risposte

client = AsyncOpenAI(base_url=ENDPOINT, api_key=os.environ["AZURE_OPENAI_API_KEY"])
enc = tiktoken.get_encoding("o200k_base")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG, encoding="utf-8"), logging.StreamHandler()],
)
log = logging.getLogger("run")

In [2]:
# ============ CELLA 2 — Prompt ============
SYSTEM = """You are a trained news-reliability analyst applying NewsGuard's rating methodology.

TASK: You are given the full text of a single news article (no metadata, no URL, no byline).
Based ONLY on this text, estimate the reliability LEVEL of the PUBLISHER that produced it,
using NewsGuard's 0-100 reliability scale discretized into five ordered levels.

NewsGuard rates a publisher against nine apolitical journalistic criteria (points sum to 100):
Credibility - (1) does not repeatedly publish false content (22); (2) gathers and presents
information responsibly (18); (3) regularly corrects or clarifies errors (12.5); (4) handles
the difference between news and opinion responsibly (12.5); (5) avoids deceptive headlines (10).
Transparency - (6) discloses ownership and financing (7.5); (7) clearly labels advertising (7.5);
(8) reveals who is in charge, including possible conflicts of interest (5); (9) provides the names
of content creators, with contact or biographical information (5).

Map your judgement to one of five ordered reliability levels:
5 = score 100 (High Credibility): adheres to all nine criteria.
4 = score 75-99 (Generally Credible): adheres to most criteria; only minor lapses.
3 = score 60-74 (Credible but with Exceptions): generally credible but fails some criteria.
2 = score 40-59 (Proceed with Caution): serious credibility or transparency problems.
1 = score 0-39 (Proceed with Maximum Caution): severe, repeated failures; generally unreliable.

Judge only from signals present in the article text: responsible vs. false/unsupported claims,
sensational or deceptive framing, mixing of news and opinion, sourcing, tone, and any
correction/transparency cues. A single article is partial evidence of its publisher; give your
best estimate of the publisher's overall level.

Output: respond with a SINGLE digit from 1 to 5 and NOTHING else."""

def build_prompt(text):
    art = str(text)
    if CAP is not None:
        art = enc.decode(enc.encode(art)[:CAP])
    return f"{SYSTEM}\n\n=== ARTICLE TEXT ===\n{art}\n\n=== YOUR ANSWER (a single digit 1-5) ==="

In [ ]:
# ============ CELLA 3 — Motore robusto (resumabile) ============
def load_checkpoint():
    if os.path.exists(OUT):
        prev = pd.read_csv(OUT)
        return {int(r.idx): (None if pd.isna(r.pred_1_5) else int(r.pred_1_5)) for r in prev.itertuples()}
    return {}

def save_checkpoint(df_src, results):
    rows = [{"idx": i,
             "domain": df_src.at[i, "domain"],
             "label": int(df_src.at[i, "label"]),
             "pred_1_5": results[i]} for i in sorted(results)]
    tmp = OUT + ".tmp"
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, OUT)          # scrittura atomica: il checkpoint non si corrompe se interrompi

async def classify(idx, text):
    prompt = build_prompt(text)
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            r = await asyncio.wait_for(
                client.responses.create(model=MODEL, input=prompt),   # SOLO model + input
                timeout=REQ_TIMEOUT,
            )
            out = (r.output_text or "").strip()
            m = re.search(r"[1-5]", out)
            if m:
                return idx, int(m.group()), None
            raise ValueError(f"output non parsabile: {out!r}")
        except Exception as e:
            wait = min(2 ** attempt, 30)
            log.warning(f"idx={idx} tent.{attempt}/{MAX_ATTEMPTS} KO: {type(e).__name__}: {e} -> attendo {wait}s")
            await asyncio.sleep(wait)
    return idx, None, "FAILED_AFTER_RETRIES"

async def main():
    df = pd.read_csv(CSV_IN).reset_index(drop=True)
    results = load_checkpoint()
    todo = [i for i in df.index if results.get(i) is None]   # i None verranno riprovati a un nuovo lancio
    log.info(f"Totale={len(df)} | già validi={len(df)-len(todo)} | da fare={len(todo)}")

    sem = asyncio.Semaphore(CONCURRENCY)
    processed = 0

    async def worker(i):
        nonlocal processed
        async with sem:
            idx, pred, err = await classify(i, df.at[i, "text"])
            results[idx] = pred
            processed += 1
            if err:
                log.error(f"idx={idx} NON classificato ({err})")
            if processed % SAVE_EVERY == 0:
                save_checkpoint(df, results)
                validi = sum(1 for v in results.values() if v is not None)
                log.info(f"progress: {processed}/{len(todo)} in questo run | validi totali={validi}")

    B = 500
    for s in range(0, len(todo), B):
        await asyncio.gather(*(worker(i) for i in todo[s:s+B]))
        save_checkpoint(df, results)
        log.info(f"checkpoint salvato ({min(s+B, len(todo))}/{len(todo)})")

    save_checkpoint(df, results)
    ok = sum(1 for v in results.values() if v is not None)
    ko = len(df) - ok
    log.info(f"FINE. Validi={ok} | mancanti={ko}." + (f" Rilancia questa cella per riprovare i {ko} mancanti." if ko else ""))

# --- Avvio ---
await main()                    # in Jupyter
# asyncio.run(main())           # se lo esegui come script .py

2026-09-29 15:36:35,701 | INFO | Totale=19476 | già validi=0 | da fare=19476
2026-09-29 15:36:41,498 | INFO | HTTP Request: POST https://daily-digest-mini-resource.services.ai.azure.com/openai/v1/responses "HTTP/1.1 200 OK"
2026-09-29 15:36:41,670 | INFO | HTTP Request: POST https://daily-digest-mini-resource.services.ai.azure.com/openai/v1/responses "HTTP/1.1 200 OK"
2026-09-29 15:36:42,269 | INFO | HTTP Request: POST https://daily-digest-mini-resource.services.ai.azure.com/openai/v1/responses "HTTP/1.1 200 OK"
2026-09-29 15:36:42,395 | INFO | HTTP Request: POST https://daily-digest-mini-resource.services.ai.azure.com/openai/v1/responses "HTTP/1.1 200 OK"
2026-09-29 15:36:42,723 | INFO | HTTP Request: POST https://daily-digest-mini-resource.services.ai.azure.com/openai/v1/responses "HTTP/1.1 200 OK"
2026-09-29 15:36:43,383 | INFO | HTTP Request: POST https://daily-digest-mini-resource.services.ai.azure.com/openai/v1/responses "HTTP/1.1 200 OK"
2026-09-29 15:36:43,924 | INFO | HTTP Req

# ENRICH

In [ ]:
import pandas as pd

CSV_IN = "Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv"
OUT    = "preds_gpt5mini_capNone.csv"          # il checkpoint del run in corso

src = pd.read_csv(CSV_IN).reset_index(drop=True)   # STESSO caricamento del run -> idx allineato
src["idx"] = src.index                              # idx = chiave di join

preds = pd.read_csv(OUT)                             # idx, domain, label, pred_1_5

# tutte le colonne del dataset TRANNE il testo, + la predizione
meta = src[[c for c in src.columns if c != "text"]]      # url, title, authors, domain, score, ...
enriched = meta.merge(preds[["idx", "pred_1_5"]], on="idx", how="left")
enriched["pred_0_4"] = enriched["pred_1_5"] - 1          # allineato alla colonna 'label'

enriched.to_csv("preds_gpt5mini_enriched.csv", index=False)
print("righe:", len(enriched), "| valide:", enriched["pred_1_5"].notna().sum())
enriched[["idx","url","domain","label","pred_1_5"]].head()

# VALUTAZIONE

In [ ]:
# ============ CELLA 4 — Metriche (domani) ============
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def tolerant_accuracy(y_true, y_pred, alpha):
    d = np.abs(np.asarray(y_true) - np.asarray(y_pred))
    return np.where(d == 0, 1.0, np.where(d == 1, alpha, 0.0)).mean()

d = pd.read_csv(OUT).dropna(subset=["pred_1_5"]).copy()
d["pred"] = d["pred_1_5"].astype(int) - 1          # 1-5 -> 0-4, allineato a 'label'

for name, t, p in [("Article", d.label, d.pred)]:
    print(f"== {name} ==  Acc={accuracy_score(t,p):.4f}  F1={f1_score(t,p,average='macro'):.4f}  "
          f"Tol@0.5={tolerant_accuracy(t,p,0.5):.4f}  Tol@1.0={tolerant_accuracy(t,p,1.0):.4f}")

pub = d.groupby("domain").agg(true=("label","first"),
                              pred=("pred", lambda s: s.mode().iloc[0])).reset_index()
print(f"== Publisher ==  Acc={accuracy_score(pub.true,pub.pred):.4f}  F1={f1_score(pub.true,pub.pred,average='macro'):.4f}  "
      f"Tol@0.5={tolerant_accuracy(pub.true,pub.pred,0.5):.4f}  Tol@1.0={tolerant_accuracy(pub.true,pub.pred,1.0):.4f}")